In [7]:
import pandas as pd
import numpy as np

def change_axis(x):
    if x == "[0 1 0]":
        return "Y"
    if x == "[0 0 1]":
        return "Z"

In [8]:
# Model data and Ganis + Kievet data

df_gk = pd.read_csv('GanisKievet_cleaned.csv')
df_model = pd.read_csv('../rotation_model_data_gk.csv')

# move/add/delete columns
df_gk = df_gk.drop(columns=['age', 'trial_num', 'condition'])
df_gk['false_positive'] = np.where(((df_gk['correct'] == 0) & (df_gk['response'] == '[b]')), 1, 0)
df_gk['false_negative'] = np.where(((df_gk['correct'] == 0) & (df_gk['response'] == '[n]')), 1, 0)
df_gk['source'] = "gk"
df_gk['axis'] = "Y"
df_gk['axis_difficulty'] = 0
df_gk['num_of_repeats'] = None
df_gk['encoding_time'] = None
df_gk['landmarking_time'] = None
df_gk['rotation_time'] = None
df_gk['decision_time'] = None

# fix data format
df_gk["gender"] = df_gk["gender"].map({
    "F": "female",
    "M": "male"
})
df_gk["response"] = df_gk["response"].map({
    "[n]": "different",
    "[b]": "same"
})
df_gk["expected_answer"] = df_gk["expected_answer"].map({
    "[n]": "different",
    "[b]": "same"
})
df_gk["time"] = df_gk["time"]/1000.0

df_model['subject_num'] = df_model['subject_num'] + df_gk['subject_num'].max()      # all diff subjects between experiments
df_model["axis"] = df_model["axis"].apply(lambda x: change_axis(x))

# combine dfs
df_combined =  pd.concat([df_gk, df_model], axis=0, ignore_index=True)
print(df_combined)
df_combined.to_csv('gk_model_data_new.csv', index=False)

      subject_num  gender  angle   time   response expected_answer  correct  \
0               1    male    0.0  1.355  different       different        1   
1               1    male  150.0  2.079  different       different        1   
2               1    male  150.0  1.834       same            same        1   
3               1    male  100.0  4.780       same            same        1   
4               1    male   50.0  1.685       same            same        1   
...           ...     ...    ...    ...        ...             ...      ...   
9888          104  female    0.0  1.418       same            same        1   
9889          104  female    0.0  1.418       same            same        1   
9890          104  female  150.0  4.843  different       different        1   
9891          104  female   50.0  2.704  different       different        1   
9892          104  female  100.0  2.384       same            same        1   

      false_positive  false_negative source axis  a

/tmp/ipykernel_112657/3097113972.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined =  pd.concat([df_gk, df_model], axis=0, ignore_index=True)


In [9]:
# Jost + Jonsen data

df_jj = pd.read_csv('JostJansen_cleaned.csv')
df_model = pd.read_csv('../rotation_model_data_jj.csv')

# move/add/delete columns
df_jj = df_jj.drop(columns=['ID', 'model', 'type', 'number', 'absTime', 'modelNumber', 'correctSide', "outlier", 'typeOutlier', 'originalDegrees', 'direction', 'endTime', 'startTime', 'pauseTime', 'MRexperience'])
df_jj['expected_answer'] = np.where(df_jj['orientationLeftBase'] == df_jj['orientation'], "same", "different")
df_jj['response'] = np.where(((df_jj['expected_answer'] == "same") & (df_jj['correctAnswer'] == "hit")) | ((df_jj['expected_answer'] == "different") & (df_jj['correctAnswer'] == "incorrect")), "same", "different")
df_jj['expected_answer'] = df_jj.pop('expected_answer')
df_jj['correct'] = df_jj.pop('correctAnswer')
df_jj.insert(1, "gender", df_jj.pop('Gender'))
df_jj.insert(2, "angle", df_jj.pop('deg'))
df_jj.insert(3, "axis", df_jj.pop('axis'))
df_jj.insert(4, "time", df_jj.pop('reactionTime'))
df_jj.pop('orientation')
df_jj.pop('orientationLeftBase')
df_jj['false_positive'] = np.where(((df_jj['correct'] == "incorrect") & (df_jj['response'] == 'same')), 1, 0)
df_jj['false_negative'] = np.where(((df_jj['correct'] == "incorrect") & (df_jj['response'] == 'different')), 1, 0)
df_jj['source'] = "jj"

df_jj['axis_difficulty'] = 0
df_jj['num_of_repeats'] = None
df_jj['encoding_time'] = None
df_jj['landmarking_time'] = None
df_jj['rotation_time'] = None
df_jj['decision_time'] = None

# rename columns
df_jj = df_jj.rename(columns={
    "block": "subject_num", 
    "deg": "angle",
    "reactionTime": "time",

})

# fix data format
df_jj['subject_num'] = df_jj['subject_num'].str.replace("id", "").astype(int)
df_jj["gender"] = df_jj["gender"].map({
    "f": "female",
    "m": "male"
})
df_jj["correct"] = df_jj["correct"].map({
    "hit": 1,
    "incorrect": 0
})
df_jj["time"] = df_jj["time"]/1000.0

df_model['subject_num'] = df_model['subject_num'] + df_jj['subject_num'].max()      # all diff subjects between experiments
df_model["axis"] = df_model["axis"].apply(lambda x: change_axis(x))

# combine dfs
df_combined =  pd.concat([df_jj, df_model], axis=0, ignore_index=True)
print(df_combined)
df_combined.to_csv('jj_model_data_new.csv', index=False)

       subject_num  gender  angle axis    time   response expected_answer  \
0               23    male    135    Z  4.5199  different       different   
1               23    male     90    Z  3.1190  different       different   
2               23    male    135    Z  2.4763  different       different   
3               23    male    180    Z  3.6040       same            same   
4               23    male     45    Z  1.5606  different       different   
...            ...     ...    ...  ...     ...        ...             ...   
35815           81  female     45    Y  2.1510  different            same   
35816           81  female    135    Y  2.5850       same            same   
35817           81  female     45    Z  1.8570       same            same   
35818           81  female      0    Z  1.5660  different       different   
35819           81  female      0    Y  1.5660       same       different   

       correct  false_positive  false_negative source  axis_difficulty  \
0

/tmp/ipykernel_112657/830730594.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_combined =  pd.concat([df_jj, df_model], axis=0, ignore_index=True)
